# 실습 ② 요약 생성 — 연습본

> **[권장 난이도: 중급]** · **[추천 소요 시간: 100분]** · **[교육 과정: 12~13주차 · 5~6교시]**

4교시에서 꺼낸 텍스트·표·이미지를 **검색 가능한 요약문**으로 바꿉니다.

---

## 오늘 만드는 것

```
텍스트 ──┐
  표   ──┼──▶  요약문 생성  ──▶  검색용 색인(색인 카드)
이미지 ──┘      LLM / VLM
```

## 진행 방법

1. 셀을 **위에서부터 순서대로** 실행합니다 (`Shift + Enter`)
2. `# TODO` 와 `____` 가 있는 곳을 직접 채웁니다
3. 각 단계의 **확인 포인트**를 통과하면 다음으로 넘어갑니다
4. 막히면 바로 손을 드세요

## ⚠️ 이번 실습부터 API 비용이 발생합니다

`gpt-4o-mini` 기준으로 PDF 한 편에 수십 원 수준입니다. 다만 아래 두 가지는 조심하세요.

- 요약 셀을 **반복 실행하면 그만큼 비용이 다시 나갑니다**
- `max_concurrency`를 함부로 올리면 요청 제한(`RateLimitError`)에 걸립니다

## 0. API 키 불러오기

`.env` 파일에서 키를 읽습니다.

> 💡 키 전체를 `print()`로 찍지 마세요. 화면 공유나 스크린샷 한 번이면 그대로 유출됩니다.

In [ ]:
import os
import time
import pickle
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# .env 파일에서 API 키를 불러옵니다
load_dotenv(PROJECT_ROOT / ".env")
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OpenAI API 키가 없습니다. .env 파일을 확인하세요.")

print("API 키 로드 완료:", openai_api_key[:7] + "..." + openai_api_key[-4:])

> ✅ **확인 포인트:** `API 키 로드 완료: sk-proj...abcd` 처럼 앞뒤 일부가 출력되면 성공입니다.
>
> 에러가 나면 `.env` 파일이 프로젝트 루트에 있는지, 키 앞뒤에 따옴표나 공백이 없는지 확인하세요.

## 1. 4교시 결과 복원

추출에 2~3분이 걸리므로 다시 돌리지 않고 저장해 둔 파일을 불러옵니다.

> ⚠️ `FileNotFoundError`가 나면 4교시 마지막 저장 셀을 실행하지 않은 것입니다. 4교시 노트북으로 돌아가 저장 셀만 다시 돌리세요.

In [ ]:
pkl_path = PROJECT_ROOT / "data" / "processed" / "extracted.pkl"

with open(pkl_path, "rb") as f:
    data = pickle.load(f)

texts = data["texts"]
tables = data["tables"]
texts_2k_token = data["texts_2k_token"]

print("불러오기 완료:", pkl_path)
print("  텍스트 블록 :", len(texts))
print("  표 블록     :", len(tables))
print("  분할 텍스트 :", len(texts_2k_token))

# 이미지 폴더 경로도 다시 잡아둡니다
figures_dir = PROJECT_ROOT / "data" / "figures"
print("  이미지 폴더 :", figures_dir, "(존재:", figures_dir.exists(), ")")

## 2. 요약 체인의 구조

LCEL 파이프(`|`)는 **컨베이어 벨트**입니다. 왼쪽 작업대의 결과물이 오른쪽 작업대의 재료가 됩니다.

```
{"element": lambda x: x}  →  prompt  →  model  →  StrOutputParser()
       입력 포장              템플릿 적용   LLM 호출      문자열로 꺼내기
   "원본 텍스트"          "요약해줘: 원본"   AIMessage      "요약문"
```

| 칸 | 받는 것 | 내보내는 것 |
| --- | --- | --- |
| `{"element": lambda x: x}` | 문자열 | `{"element": 문자열}` 딕셔너리 |
| `prompt` | 딕셔너리 | 완성된 프롬프트 |
| `model` | 프롬프트 | `AIMessage` 객체 |
| `StrOutputParser()` | `AIMessage` | 순수 문자열 |

**TODO 1~3.** 아래 함수의 빈칸을 채우세요.

💡 **힌트**
- TODO 1: 교재의 `gpt-4`는 비쌉니다. 비전 기능까지 포함된 저렴한 최신 모델 이름입니다 (`gpt-4o-mini`)
- TODO 2, 3: 위 컨베이어 벨트 그림의 2번, 3번 칸에 들어갈 변수 이름입니다

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


def generate_text_summaries(texts, tables, summarize_texts=False):
    """
    텍스트 및 표 데이터를 요약하여 검색에 활용할 요약본을 생성합니다.

    texts          : 텍스트 리스트
    tables         : 표 리스트
    summarize_texts: 텍스트 요약을 활성화할지 여부
    """

    prompt_text_kor = """당신은 표와 텍스트를 요약하여 검색에 활용할 수 있도록 돕는 도우미입니다.
이 요약본들은 임베딩되어 원본 텍스트나 표 요소를 검색하는 데 사용될 것입니다.
주어진 표나 텍스트의 내용을 검색에 최적화된 간결한 요약으로 작성해주세요.
요약할 표 또는 텍스트: {element}"""

    prompt = ChatPromptTemplate.from_template(prompt_text_kor)

    # TODO 1: 사용할 모델 이름을 적으세요
    model = ChatOpenAI(temperature=0, model="____")

    # TODO 2, 3: 컨베이어 벨트의 2번, 3번 칸을 채우세요
    summarize_chain = {"element": lambda x: x} | ____ | ____ | StrOutputParser()

    text_summaries = []
    table_summaries = []

    if texts and summarize_texts:
        text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})
    elif texts:
        text_summaries = texts

    if tables:
        table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

    return text_summaries, table_summaries


print("generate_text_summaries 정의 완료")

## 3. 텍스트·표 요약 실행

> ⏱️ **API를 호출하므로 1~3분 걸립니다.** 블록 수에 비례합니다.
>
> 💸 이 셀을 반복 실행하면 그만큼 비용이 다시 나갑니다. 한 번만 돌리세요.

In [ ]:
start = time.time()

text_summaries, table_summaries = generate_text_summaries(
    texts_2k_token, tables, summarize_texts=True
)

print(f"요약 완료 — {time.time() - start:.1f}초 소요")
print("텍스트 요약 수:", len(text_summaries))
print("표 요약 수    :", len(table_summaries))

## 4. 요약 결과 확인

원본과 요약을 나란히 놓고 비교합니다.

In [ ]:
print("=" * 70)
print("[원본] texts_2k_token[0] — 앞 300자")
print("=" * 70)
print(texts_2k_token[0][:300])
print()

print("=" * 70)
print("[요약] text_summaries[0]")
print("=" * 70)
print(text_summaries[0])
print()

orig = len(texts_2k_token[0])
summ = len(text_summaries[0])
print(f"압축률: {orig}자 → {summ}자 ({summ / orig * 100:.1f}%)")

> ✅ **확인 포인트:** 요약문이 한국어로 출력되고, 원본의 핵심 내용을 담고 있으면 성공입니다.

## 5. `table_summaries`가 비어 있다면

**비어 있어도 정상일 수 있습니다.** 코드가 틀린 게 아닙니다.

원인은 두 가지입니다.

1. 해당 PDF에 **표 객체가 없음** — 표가 이미지로 그려져 있으면 표로 잡히지 않습니다
2. 4교시에서 `strategy="hi_res"` **누락**

아래 셀이 원인을 자동 진단합니다.

In [ ]:
if table_summaries:
    print(f"표 요약 {len(table_summaries)}건 생성됨\n")
    print("[표 요약 예시]")
    print(table_summaries[0])
else:
    print("표 요약이 비어 있습니다. 아래를 확인하세요.\n")
    print("  1) 4교시에서 strategy='hi_res' 를 넣었는가?")
    print("  2) PDF를 열어 표가 '진짜 표'인지, '그림으로 그린 표'인지 확인했는가?")
    print("     → 그림으로 된 표는 이미지로 추출되며, 6교시 이미지 요약이 처리합니다.")
    print("\n  둘 다 해당하지 않으면 정상입니다. 그대로 진행하세요.")

**TODO 4. 내 PDF는 어느 경우인가요? (더블클릭해서 편집)**

- 표 요약 개수:
- 비어 있다면 그 이유는:

---

# 6교시 — 이미지 요약

여기서부터 **VLM(Vision Language Model)** 이 등장합니다. 글자만 읽던 모델에 눈을 달아주는 단계입니다.

## 6. 이미지를 Base64로 포장하기

> **비유:** Base64는 **그림을 택배로 보내기 위해 '문자열 상자'에 포장하는 작업**입니다. API는 문자열만 주고받을 수 있으니, 이미지를 글자로 바꿔서 보냅니다.

**TODO 5.** 인코딩 결과를 어떤 문자 인코딩으로 디코딩할지 채우세요.

💡 **힌트:** 전 세계 문자를 다루는 표준 인코딩입니다. 보통 `"utf-8"`을 씁니다.

In [ ]:
import base64

from langchain_core.messages import HumanMessage


def encode_image(image_path):
    """이미지를 base64 문자열로 변환"""
    with open(image_path, "rb") as image_file:
        # TODO 5: 어떤 인코딩으로 디코딩할까요?
        return base64.b64encode(image_file.read()).decode("____")


image_files = sorted(
    f for f in figures_dir.iterdir()
    if f.suffix.lower() in (".jpg", ".jpeg", ".png")
)

sample_b64 = encode_image(image_files[0])

print("원본 파일 :", image_files[0].name)
print("문자열 길이:", len(sample_b64), "자")
print("앞 100자  :", sample_b64[:100])

> ✅ **확인 포인트:** 알파벳과 숫자가 뒤섞인 긴 문자열이 출력되면 성공입니다. **그림이 글자가 된 순간**입니다.

## 7. 이미지 요약 함수 만들기

**TODO 6~7.** 빈칸 2개를 채우세요.

💡 **힌트**
- TODO 6: 교재의 `gpt-4-vision-preview`는 **지원이 종료된 모델**입니다. 그대로 쓰면 `404 model not found`가 납니다. 2번 셀에서 쓴 것과 같은 모델을 쓰세요
- TODO 7: `data:image/jpeg;base64,` 뒤에는 무엇이 와야 할까요? 함수가 받은 인자 이름입니다

In [ ]:
def image_summarize(img_base64, prompt):
    """이미지 요약 생성 (VLM 호출)"""
    # TODO 6: 비전 기능이 있는 모델 이름을 적으세요
    chat = ChatOpenAI(model="____", max_tokens=1024)

    msg = chat.invoke([
        HumanMessage(content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                # TODO 7: 접두사 뒤에 들어갈 변수 이름을 적으세요
                "image_url": {"url": f"data:image/jpeg;base64,{____}"},
            },
        ])
    ])
    return msg.content


test_prompt = """You are an assistant tasked with summarizing images for retrieval.
These summaries will be embedded and used to retrieve the raw image.
Provide a concise summary of the image that is well optimized for retrieval.
The summary should be written in Korean (Hangul)."""

print(image_summarize(sample_b64, test_prompt))

> ✅ **확인 포인트:** 그림을 설명하는 **한국어 문장**이 출력되면 성공입니다.
>
> 영어로 나왔다면 프롬프트 마지막 줄(`should be written in Korean`)이 빠지지 않았는지 확인하세요.

## 8. 폴더 내 전체 이미지 요약

**TODO 8.** 처리할 이미지 확장자를 채우세요.

💡 **힌트:** 4교시에서 추출된 파일들의 확장자입니다. 튜플로 여러 개를 적습니다.

> ⏱️ 이미지 수에 비례해 시간이 걸립니다. 10장이면 30초~1분 정도입니다.

In [ ]:
def generate_img_summaries(path):
    """이미지의 요약과 Base64 인코딩 문자열을 생성"""
    img_base64_list = []
    image_summaries = []

    prompt_kor = """You are an assistant tasked with summarizing images for retrieval.
These summaries will be embedded and used to retrieve the raw image.
Provide a concise summary of the image that is well optimized for retrieval.
The summary should be written in Korean (Hangul)."""

    for img_file in sorted(os.listdir(path)):
        # TODO 8: 처리할 확장자를 튜플로 적으세요
        if img_file.endswith(____):
            img_path = os.path.join(path, img_file)
            base64_image = encode_image(img_path)
            img_base64_list.append(base64_image)
            image_summaries.append(image_summarize(base64_image, prompt_kor))

    return img_base64_list, image_summaries


start = time.time()
img_base64_list, image_summaries = generate_img_summaries(str(figures_dir))

print(f"이미지 요약 완료 — {time.time() - start:.1f}초 소요")
print("요약된 이미지 수:", len(image_summaries))

## 9. 요약 품질 검증 (가장 중요한 단계)

요약이 원본을 잘못 설명하면 **검색 자체가 틀어집니다.** 그림과 요약문을 나란히 놓고 눈으로 대조하세요.

In [ ]:
from IPython.display import Image, display

for i, (img_file, summary) in enumerate(zip(image_files[:3], image_summaries[:3])):
    print("=" * 70)
    print(f"[{i}] {img_file.name}")
    print("=" * 70)
    display(Image(filename=str(img_file), width=450))
    print("요약:", summary)
    print()

### 요약 품질 기준

| 수준 | 예시 | 검색 가능성 |
| --- | --- | --- |
| 나쁨 | "차트 이미지입니다" | 거의 불가능 |
| 보통 | "매출 관련 그래프" | 낮음 |
| 좋음 | "2021~2023년 분기별 매출 추이 선그래프, 2023년 하반기 하락세" | 높음 |

**TODO 9. 내 요약은 어느 수준인가요? (더블클릭해서 편집)**

- 수준 (나쁨 / 보통 / 좋음):
- 그렇게 판단한 이유:
- 개선하려면 프롬프트에 무엇을 추가해야 할까:

> 💡 품질이 나쁘면 프롬프트에 **"축 이름, 수치 범위, 추세를 반드시 포함하세요"** 같은 지시를 추가하고 다시 돌려보세요.

## 10. 토큰 사용량 측정 (과제 가산점)

과제 가산점에 "비용 측정"이 있습니다. 여기서 나온 수치를 README에 기록하면 그대로 근거 자료가 됩니다.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

input_tokens = sum(len(enc.encode(t)) for t in texts_2k_token)
output_tokens = sum(len(enc.encode(s)) for s in text_summaries)

print("텍스트 요약 토큰 사용량 (추정)")
print(f"  입력 : {input_tokens:,} 토큰")
print(f"  출력 : {output_tokens:,} 토큰")
print(f"  합계 : {input_tokens + output_tokens:,} 토큰")
print()
print("요약한 이미지 수:", len(image_summaries), "장")
print()
print("※ 실제 비용 = 토큰 수 x 단가. 단가는 OpenAI 공식 가격 페이지에서 확인하세요.")
print("※ 이미지 토큰은 해상도에 따라 달라지므로 위 수치에 포함되지 않았습니다.")

## 11. 7교시를 위한 저장

요약 생성에도 API 비용과 시간이 듭니다. 매번 다시 돌리지 않도록 저장해 둡니다.

In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "summaries.pkl", "wb") as f:
    pickle.dump(
        {
            "text_summaries": text_summaries,
            "table_summaries": table_summaries,
            "image_summaries": image_summaries,
            "img_base64_list": img_base64_list,
        },
        f,
    )

print("저장 완료:", output_dir / "summaries.pkl")

---

## ✅ 체크포인트 2 — 모두 체크되면 5~6교시 완료

- [ ] `text_summaries[0]`에 한국어 요약문이 출력된다
- [ ] `image_summaries[0]`에 그림을 설명하는 한국어 문장이 출력된다
- [ ] 그림과 요약문을 나란히 놓고 **육안으로 검증**했다
- [ ] `table_summaries`가 비어도 그 이유를 설명할 수 있다
- [ ] `data/processed/summaries.pkl` 파일이 생성되었다
- [ ] TODO 1~9를 모두 채웠다

## 생각해 볼 질문 (답을 적어보세요)

더블클릭해서 편집하세요. 정답을 맞히는 것보다 **본인 말로 써 보는 것**이 중요합니다.

**Q1. 이미지를 왜 굳이 텍스트로 요약할까요? 이미지 그대로 검색하면 안 되나요?**

답:

**Q2. 요약문만 저장하고 원본은 버리면 어떤 문제가 생길까요?**

답:

**Q3. `max_concurrency`를 100으로 올리면 어떤 일이 벌어질까요?**

답:

## 자주 나는 에러

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| `AuthenticationError` | `.env` 키 오타 / `load_dotenv()` 누락 | 키 앞뒤 공백·따옴표 제거 |
| `404 model not found` | 지원 종료된 모델명 사용 | TODO 1, TODO 6 확인 |
| `RateLimitError` | 동시 요청 과다 | `max_concurrency`를 5 → 2로 낮추기 |
| `FileNotFoundError: extracted.pkl` | 4교시 저장 셀 미실행 | 4교시 노트북 마지막 셀 재실행 |
| `IndexError: list index out of range` | `figures/`에 이미지가 없음 | 4교시 추출 옵션 재확인 |
| 요약이 영어로 나옴 | 프롬프트에 한국어 지시 누락 | 프롬프트 마지막 줄 확인 |
| `NameError: texts_2k_token` | 1번 셀을 안 돌림 | 위에서부터 순서대로 재실행 |

> 그래도 안 되면 **커널 재시작 후 위에서부터 전체 재실행**하세요.